# Sprite Generator - Complete GPU Training
Trains VQ-VAE (50 epochs) + Transformer (50 epochs) end-to-end.
Loads dataset from HF Hub, pushes checkpoints back.

In [ ]:
import os, sys, torch, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

!pip install -q datasets huggingface_hub torchvision pillow tqdm

# Clone the public repo for model source code
if not os.path.exists('/kaggle/working/sprite-gen'):
    !git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_DATASET = "darklord8777/sprites"
HF_MODEL_REPO = "darklord8777/sprite-generator-model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"HF_TOKEN set: {bool(HF_TOKEN)}")

# Step 1: Dataset & Dataloader

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm

class SpriteDataset(torch.utils.data.Dataset):
    def __init__(self, hf_path, split="train", image_size=32):
        self.dataset = load_dataset(hf_path, split=split)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        img = item["image"].convert("RGBA")
        return self.transform(img)

ds = SpriteDataset(HF_DATASET)
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)
print(f"Dataset: {len(ds)} sprites, {len(dl)} batches/epoch")

# Step 2: Train VQ-VAE

In [ ]:
import torch.nn as nn
from models.vqvae.model import VQVAE

vqvae = VQVAE(in_channels=4, hidden_dim=128, latent_dim=64, num_embeddings=256).to(device)
optimizer = torch.optim.Adam(vqvae.parameters(), lr=3e-4)

ckpt_dir_vqvae = Path("/kaggle/working/checkpoints/vqvae")
ckpt_dir_vqvae.mkdir(parents=True, exist_ok=True)

VQVAE_EPOCHS = 50
for epoch in range(VQVAE_EPOCHS):
    vqvae.train()
    total_loss = 0
    pbar = tqdm(dl, desc=f"VQVAE Epoch {epoch+1}/{VQVAE_EPOCHS}")
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = vqvae(batch)
        out["loss"].backward()
        optimizer.step()
        total_loss += out["loss"].item()
        pbar.set_postfix({"loss": out["loss"].item(), "recon": out["recon_loss"].item(), "vq": out["vq_loss"].item()})
    avg_loss = total_loss / len(dl)
    print(f"VQVAE Epoch {epoch+1}: loss={avg_loss:.6f}")
    ckpt_path = ckpt_dir_vqvae / f"vqvae_epoch_{epoch+1:03d}.pt"
    torch.save({"epoch": epoch, "model_state": vqvae.state_dict(), "optimizer_state": optimizer.state_dict(), "loss": avg_loss}, ckpt_path)

torch.save({"epoch": VQVAE_EPOCHS-1, "model_state": vqvae.state_dict(), "optimizer_state": optimizer.state_dict(), "loss": avg_loss}, ckpt_dir_vqvae / "vqvae_final.pt")
print("VQ-VAE training complete!")

# Step 3: Push VQ-VAE Checkpoint to HF Hub

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=str(ckpt_dir_vqvae / "vqvae_final.pt"),
        path_in_repo="vqvae_latest.pt",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    print("VQ-VAE checkpoint pushed to HF Hub")
else:
    print("Skipping VQ-VAE push: no HF_TOKEN")

# Step 4: Encode all sprites through VQ-VAE to get token sequences

In [ ]:
vqvae.eval()
dataset_hf = load_dataset(HF_DATASET, split="train")

CLASS_VOCAB = ["unknown","character","item","tile","enemy","player","weapon","food",
               "vehicle","building","decoration","effect","projectile","animal","plant",
               "furniture","tool","accessory","ui_element","terrain"]
ACTION_VOCAB = ["idle","walk","run","attack","jump","hurt","death","block","shoot",
                "cast","interact","fly","swim","climb"]
DIRECTION_VOCAB = ["front","back","left","right","front_left","front_right","back_left","back_right"]

def encode_cond(value, vocab):
    try: return vocab.index(value)
    except ValueError: return 0

all_tokens, all_class, all_action, all_direction = [], [], [], []
for item in tqdm(dataset_hf, desc="Encoding tokens"):
    img = item["image"].convert("RGBA").resize((32, 32), Image.NEAREST)
    img_t = torch.tensor(np.array(img).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        tokens = vqvae.encode_to_indices(img_t).squeeze(0).cpu()
    all_tokens.append(tokens)
    all_class.append(encode_cond(item.get("class","unknown"), CLASS_VOCAB))
    all_action.append(encode_cond(item.get("action","idle"), ACTION_VOCAB))
    all_direction.append(encode_cond(item.get("direction","front"), DIRECTION_VOCAB))

token_data = {
    "tokens": torch.stack(all_tokens).numpy(),
    "class": np.array(all_class),
    "action": np.array(all_action),
    "direction": np.array(all_direction),
}
np.savez("/kaggle/working/token_dataset.npz", **token_data)
print(f"Saved {len(all_tokens)} token sequences")

# Step 5: Train Transformer Prior

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from models.transformer.model import SpriteTransformer

data = np.load("/kaggle/working/token_dataset.npz")
tokens = torch.tensor(data["tokens"])
class_ids = torch.tensor(data["class"])
action_ids = torch.tensor(data["action"])
direction_ids = torch.tensor(data["direction"])

train_ds = TensorDataset(tokens, class_ids, action_ids, direction_ids)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)

transformer = SpriteTransformer(
    vocab_size=256,
    condition_vocab_size=64,
    d_model=256,
    n_layers=8,
    n_heads=4,
    max_seq_len=tokens.shape[1] + 1,
).to(device)

optimizer_t = torch.optim.Adam(transformer.parameters(), lr=3e-4)
scheduler_t = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_t, T_max=50)

ckpt_dir_t = Path("/kaggle/working/checkpoints/transformer")
ckpt_dir_t.mkdir(parents=True, exist_ok=True)

TRANSFORMER_EPOCHS = 50
for epoch in range(TRANSFORMER_EPOCHS):
    transformer.train()
    total_loss = 0
    pbar = tqdm(train_dl, desc=f"Transformer Epoch {epoch+1}/{TRANSFORMER_EPOCHS}")
    for batch_tokens, c, a, d in pbar:
        batch_tokens, c, a, d = batch_tokens.to(device), c.to(device), a.to(device), d.to(device)
        optimizer_t.zero_grad()
        logits = transformer(batch_tokens, c, a, d)
        loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), batch_tokens.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
        optimizer_t.step()
        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
    scheduler_t.step()
    avg_loss = total_loss / len(train_dl)
    print(f"Transformer Epoch {epoch+1}: loss={avg_loss:.6f}")
    if (epoch + 1) % 10 == 0:
        torch.save({"epoch": epoch, "model_state": transformer.state_dict(), "optimizer_state": optimizer_t.state_dict(), "loss": avg_loss}, ckpt_dir_t / f"transformer_epoch_{epoch+1:03d}.pt")

torch.save({"epoch": TRANSFORMER_EPOCHS-1, "model_state": transformer.state_dict(), "optimizer_state": optimizer_t.state_dict(), "loss": avg_loss}, ckpt_dir_t / "transformer_final.pt")
print("Transformer training complete!")

# Step 6: Push Everything to HF Hub

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    
    # Push transformer checkpoint
    api.upload_file(
        path_or_fileobj=str(ckpt_dir_t / "transformer_final.pt"),
        path_in_repo="transformer_latest.pt",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    print("Transformer checkpoint pushed")
    
    # Push training marker
    api.upload_file(
        path_or_fileobj=json.dumps({"status": "complete", "vqvae_epochs": VQVAE_EPOCHS, "transformer_epochs": TRANSFORMER_EPOCHS}).encode(),
        path_in_repo="training_complete.json",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
    )
    print("Training marker pushed")
else:
    print("Skipping HF push: no HF_TOKEN")

In [ ]:
print("=== ALL DONE ===")